# Reproducing ProcessTransformer on the Helpdesk event log
**Paper:** Bukhsh, Saeed & Dijkman (2021), *ProcessTransformer: Predictive Business Process Monitoring with Transformer Network*, arXiv:2104.00721
**Author of this notebook:** Jules Amimer — MSc AI for Business (NEOMA x INSA)

This notebook runs the **official code** of the authors with the smallest possible changes, on the Helpdesk dataset (4,580 cases, 21,348 events, 14 activities), for the 3 tasks of the paper:
1. next activity prediction
2. next event time prediction
3. remaining time prediction

Runtime: CPU is enough (~10–15 min per task for 100 epochs). A GPU (Runtime > Change runtime type) makes it faster.

## 1. Setup
The code was written for TensorFlow 2.4 / Keras 2. Recent TensorFlow ships Keras 3, so we install `tf-keras` (Keras 2) and tell TensorFlow to use it. **Restart the runtime if TensorFlow was already imported.**

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # use Keras 2 API, as in the original code
!pip install -q tf-keras
!git clone -q https://github.com/Zaharah/processtransformer.git
%cd processtransformer

## 2. Data
The original source is the 4TU Research Data repository (Helpdesk log, Verenich 2016). For convenience we download an identical copy (same 4,580 cases / 21,348 events / 14 activities, columns `Case ID`, `Activity`, `Complete Timestamp`) hosted on GitHub. You can also upload the CSV downloaded from 4TU.

In [ ]:
!mkdir -p datasets/helpdesk
!wget -q -O datasets/helpdesk/helpdesk.csv https://media.githubusercontent.com/media/lmu-dbs/BEST/HEAD/data/helpdesk.csv
import pandas as pd
raw = pd.read_csv("datasets/helpdesk/helpdesk.csv")
print(raw.shape, "cases:", raw["Case ID"].nunique(), "activities:", raw["Activity"].nunique())
raw.head()

## 3. Two compatibility fixes (documented in the logbook)
1. **Date parsing:** the original code uses `pd.to_datetime(..., dayfirst=True)` on timestamps like `2012/10/09`. With pandas ≥ 2 this crashes (it guesses the format `%Y-%d-%m`). With some versions it could silently swap day and month. We set `dayfirst=False`.
2. **Data splitting:** `np.array_split(df, n)` returns NumPy arrays (not DataFrames) with NumPy ≥ 2, which breaks the preprocessing. We split row indices instead.

No change is made to the model, the training or the evaluation.

In [ ]:
p = "processtransformer/data/processor.py"
s = open(p).read()
s = s.replace('dayfirst=True)', 'dayfirst=False)')
s = s.replace('np.array_split(df, self._pool)',
              '[df.iloc[ix] for ix in np.array_split(np.arange(len(df)), self._pool)]')
open(p, "w").write(s)
!grep -n "dayfirst\|array_split" processtransformer/data/processor.py

## 4. Preprocessing (official script)

In [ ]:
for task in ["next_activity", "next_time", "remaining_time"]:
    !python data_processing.py --dataset=helpdesk --task={task}

## 5. Training and evaluation (official scripts, README commands: 100 epochs)
Default hyperparameters of the code are kept: batch size 12, learning rate 0.001.

In [ ]:
!python next_activity.py --dataset=helpdesk --epochs=100

In [ ]:
!python next_time.py --dataset=helpdesk --epochs=100

In [ ]:
!python remaining_time.py --dataset=helpdesk --epochs=100

## 6. Results vs paper
⚠️ The official script appends a summary row at the end of each results CSV, and for next activity this row contains the **last** prefix accuracy instead of the mean (bug). We recompute the averages ourselves from the per-prefix rows.

In [ ]:
import numpy as np
r = "results/helpdesk/results"
na = pd.read_csv(r + "_next_activity.csv").iloc[:-1]   # drop the summary row
nt = pd.read_csv(r + "_next_time.csv").iloc[:-1]
rt = pd.read_csv(r + "_remaining_time.csv").iloc[:-1]
test = pd.read_csv("datasets/helpdesk/processed/next_activity_test.csv")
w = test["k"].value_counts().reindex(na["k"]).values      # number of test samples per prefix length

summary = pd.DataFrame({
  "metric": ["Next activity accuracy (%)", "Next activity weighted F1", "Next event time MAE (days)", "Remaining time MAE (days)"],
  "paper": [85.63, 0.82, 2.98, 3.72],
  "ours (mean over k, paper protocol)": [na.accuracy.mean()*100, na.fscore.mean(), nt.mean_absolute_error.mean(), rt.mean_absolute_error.mean()],
  "ours (weighted by #samples)": [np.average(na.accuracy, weights=w)*100, np.average(na.fscore, weights=w), np.nan, np.nan],
})
summary.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(na.k + 1, w, color="#ddd", label="# test samples")
ax2 = ax.twinx(); ax2.plot(na.k + 1, na.accuracy*100, "o-", color="C0", label="accuracy")
ax.set_xlabel("prefix length (number of events observed)"); ax.set_ylabel("# test samples"); ax2.set_ylabel("accuracy (%)")
plt.title("Next activity accuracy by prefix length — Helpdesk"); plt.show()

## 7. Simple baselines (my extension)
How much does the Transformer add compared with very simple frequency rules? Same train/test split, same metrics.

In [ ]:
!wget -q -O baselines.py https://raw.githubusercontent.com/jwlamr/lstm-process-monitoring-reproduction/main/extension/processtransformer/code/baselines.py
!python baselines.py datasets/helpdesk/processed results/baselines_next_activity.csv